# 1.0 – Initial Data Exploration

**Purpose**: Understand the raw dataset — image quality, lighting variance, skew distribution, and label distribution.

> ⚠️  This notebook is for **exploration only**.  
> Production logic lives in `src/`.  
> Do not import from this notebook in other code.

In [ ]:
import sys
sys.path.insert(0, '..')   # so we can import from src/

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.dataset import load_annotations_csv
from src.features.preprocess import PlatePreprocessor
from src.models.postprocess import validate_iso6346
import yaml

with open('../configs/model_config.yaml') as f:
    cfg = yaml.safe_load(f)

preprocessor = PlatePreprocessor(cfg['preprocessing'])
print('Config loaded OK')

## 1. Dataset Overview

In [ ]:
raw_dir = Path('../data/01_raw')
images  = list(raw_dir.glob('*.jpg')) + list(raw_dir.glob('*.png'))
print(f'Found {len(images)} images in {raw_dir}')

## 2. Preprocessing Visualisation

In [ ]:
# Pick a sample image and visualise each preprocessing step
if images:
    img_bgr = cv2.imread(str(images[0]))
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    deskewed = preprocessor._deskew(gray.copy())
    clahe_d  = preprocessor._apply_clahe(deskewed.copy())
    sharp_d  = preprocessor._sharpen(clahe_d.copy())

    fig, axes = plt.subplots(1, 4, figsize=(18, 3))
    for ax, (title, im) in zip(axes, [
        ('Original', gray), ('Deskewed', deskewed),
        ('CLAHE',    clahe_d), ('Sharpened', sharp_d),
    ]):
        ax.imshow(im, cmap='gray'); ax.set_title(title); ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('No images found – add some to data/01_raw/')

## 3. ISO 6346 Checksum Sanity Check

In [ ]:
test_ids = ['BICU 123456 8', 'MSCU 000000 0', 'INVALID 12345']
for raw in test_ids:
    v = validate_iso6346(raw)
    status = '✅' if v.is_valid_checksum else ('⚠️ bad checksum' if v.is_valid_format else '❌ bad format')
    print(f'{raw:25s}  {status}  normalised={v.normalised_id}')